In [64]:
import pandas as pd
import numpy as np
import sys
from pathlib import Path

# Configuración de rutas (asegúrate de que ROOT suba a la raíz)
ROOT = Path.cwd().parent
sys.path.append(str(ROOT))
from config.paths import RAW_DIR, PROCESSED_DIR

YEARS = [2018, 2024]

# comentario 

In [82]:
in_kind_transfers = pd.read_csv(PROCESSED_DIR / "in_kind_transfers.csv",
                        dtype={'folioviv': str, 'foliohog': str, 'numren': int})


iva_hogar = pd.read_csv(PROCESSED_DIR / "iva_hogar.csv",
                        dtype={'folioviv': str, 'foliohog': str})


income_fiscal_incidence_base = pd.read_csv(PROCESSED_DIR / "income_fiscal_incidence_base.csv",
                        dtype={'folioviv': str, 'foliohog': str, 'numren': int})


demographic_household = pd.read_csv(PROCESSED_DIR / "demographic_household.csv",
                        dtype={'folioviv': str, 'foliohog': str})




/var/folders/lw/wsksmjcd29scklkcylgwjjlr0000gn/T/ipykernel_75403/2214915603.py:1: DtypeWarning: Columns (0: etnia) have mixed types. Specify dtype option on import or set low_memory=False.
  in_kind_transfers = pd.read_csv(PROCESSED_DIR / "in_kind_transfers.csv",


In [83]:
import pandas as pd

# ==========================================================
# 1. Número de integrantes por hogar (según income_fiscal_incidence_base)
# ==========================================================

print("\n--- Paso 1: household_size ---")
print(f"Tamaño de income_fiscal_incidence_base: {len(income_fiscal_incidence_base)}")

household_size = (
    income_fiscal_incidence_base
    .groupby(["folioviv", "foliohog", "year"])
    .size()
    .reset_index(name="household_size")
)

print(f"Hogares encontrados: {len(household_size)}")

# ==========================================================
# 2. Convertir IVA hogar a IVA per cápita
# ==========================================================

print("\n--- Paso 2: IVA per cápita ---")
print(f"Tamaño de iva_hogar antes: {len(iva_hogar)}")

iva_hogar = iva_hogar.merge(
    household_size,
    on=["folioviv", "foliohog", "year"],
    how="inner"
)

print(f"Tamaño de iva_hogar después del merge: {len(iva_hogar)}")

# Todas las columnas de IVA (excepto llaves) se dividen entre el tamaño del hogar
iva_cols = [
    c for c in iva_hogar.columns
    if c not in ["folioviv", "foliohog", "year", "household_size"]
]

iva_hogar[iva_cols] = iva_hogar[iva_cols].div(
    iva_hogar["household_size"],
    axis=0
)


print("Variables de IVA convertidas a nivel individuo.")

# ==========================================================
# 3. Base a nivel hogar
# ==========================================================

print("\n--- Paso 3: Base a nivel hogar ---")
print(f"Tamaño de demographic_household (antes del merge): {len(demographic_household)}")
print(f"Tamaño de iva_hogar (antes del merge): {len(iva_hogar)}")

household_base = (
    demographic_household
    .merge(
        iva_hogar,
        on=["folioviv", "foliohog", "year"],
        how="inner"
    )
)

print(f"Tamaño de household_base (después del merge): {len(household_base)}")

# ==========================================================
# 4. Llevar variables del hogar a la base individual
# ==========================================================

print("\n--- Paso 4: Base individual con variables del hogar ---")
print(f"Tamaño de income_fiscal_incidence_base (antes del merge): {len(income_fiscal_incidence_base)}")
print(f"Tamaño de household_base (antes del merge): {len(household_base)}")

individual_base = (
    income_fiscal_incidence_base
    .merge(
        household_base,
        on=["folioviv", "foliohog", "year"],
        how="inner"
    )
)

print(f"Tamaño de individual_base (después del merge): {len(individual_base)}")

# ==========================================================
# 5. Agregar transferencias en especie
# ==========================================================

print("\n--- Paso 5: Agregar transferencias en especie ---")

# Asegurar que las llaves tengan el mismo tipo
individual_base["folioviv"] = individual_base["folioviv"].astype(str)
individual_base["foliohog"] = individual_base["foliohog"].astype(str)
individual_base["numren"] = individual_base["numren"].astype(str)

in_kind_transfers["folioviv"] = in_kind_transfers["folioviv"].astype(str)
in_kind_transfers["foliohog"] = in_kind_transfers["foliohog"].astype(str)
in_kind_transfers["numren"] = in_kind_transfers["numren"].astype(str)

print(f"Tamaño de individual_base (antes del merge con transfers): {len(individual_base)}")
print(f"Tamaño de in_kind_transfers (antes del merge): {len(in_kind_transfers)}")

# Merge
individual_base = individual_base.merge(
    in_kind_transfers,
    on=["folioviv", "foliohog", "numren", "year"],
    how="inner"
)

print(f"Tamaño de individual_base (después del merge con transfers): {len(individual_base)}")

# ==========================================================
# 6. Conteo por año (2018 y 2024)
# ==========================================================

print("\n--- Conteo de registros por año en la base final ---")
print(individual_base['year'].value_counts().sort_index())

# Si quieres ver específicamente 2018 y 2024:
print(f"\nRegistros para 2018: {len(individual_base[individual_base['year'] == 2018])}")
print(f"Registros para 2024: {len(individual_base[individual_base['year'] == 2024])}")

# ==========================================================
# Resultado
# ==========================================================

print("\n--- Muestra de la base final ---")
print(individual_base.head())

individual_base.to_csv(PROCESSED_DIR / "base_final.csv", index=False)
print("Saved base_final.csv")


--- Paso 1: household_size ---
Tamaño de income_fiscal_incidence_base: 384799
Hogares encontrados: 165949

--- Paso 2: IVA per cápita ---
Tamaño de iva_hogar antes: 165950
Tamaño de iva_hogar después del merge: 165902
Variables de IVA convertidas a nivel individuo.

--- Paso 3: Base a nivel hogar ---
Tamaño de demographic_household (antes del merge): 166061
Tamaño de iva_hogar (antes del merge): 165902
Tamaño de household_base (después del merge): 165902

--- Paso 4: Base individual con variables del hogar ---
Tamaño de income_fiscal_incidence_base (antes del merge): 384799
Tamaño de household_base (antes del merge): 165902
Tamaño de individual_base (después del merge): 384743

--- Paso 5: Agregar transferencias en especie ---
Tamaño de individual_base (antes del merge con transfers): 384743
Tamaño de in_kind_transfers (antes del merge): 577804
Tamaño de individual_base (después del merge con transfers): 384743

--- Conteo de registros por año en la base final ---
year
2018    182397


In [86]:
base_final = pd.read_csv(PROCESSED_DIR / "base_final.csv",
                        dtype={'folioviv': str, 'foliohog': str, 'numren': str})

base_final


/var/folders/lw/wsksmjcd29scklkcylgwjjlr0000gn/T/ipykernel_75403/3169126187.py:1: DtypeWarning: Columns (0: etnia) have mixed types. Specify dtype option on import or set low_memory=False.
  base_final = pd.read_csv(PROCESSED_DIR / "base_final.csv",


,folioviv,foliohog,numren,year,net_trabajo1_ann,net_trabajo2_ann,isr_total,formal_trabajo_1,formal_trabajo_2,mi_trabajo_1,...,iva_pagado,gasto_total,gasto_gas,gasto_electricity,household_size,sexo,edad,etnia,education,health
0,0100013601,1,1,2018,118032.76,0.0,0.0,0.0,0.0,118032.760000,...,705.752644,6183.823333,2400.0,200.0,3,1,74,1,0,1
1,0100013601,1,3,2018,94426.20,0.0,0.0,0.0,0.0,94426.200000,...,705.752644,6183.823333,2400.0,200.0,3,1,38,1,0,0
2,0100013601,1,2,2018,0.00,0.0,0.0,0.0,0.0,0.000000,...,705.752644,6183.823333,2400.0,200.0,3,2,70,2,0,1
3,0100013602,1,2,2018,0.00,5901.6,0.0,0.0,0.0,0.000000,...,1012.856000,9561.994000,720.0,90.0,5,2,47,2,0,1
4,0100013602,1,3,2018,0.00,0.0,0.0,0.0,0.0,0.000000,...,1012.856000,9561.994000,720.0,90.0,5,2,20,2,1,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
384738,3260593816,1,3,2024,32262.28,0.0,0.0,0.0,0.0,24158.561919,...,568.817931,5741.663333,2000.0,140.0,3,1,21,2.0,0,0
384739,3260593816,1,4,2024,0.00,0.0,0.0,0.0,0.0,0.000000,...,568.817931,5741.663333,2000.0,140.0,3,1,15,2.0,1,0
384740,3260593817,1,2,2024,0.00,0.0,0.0,0.0,0.0,0.000000,...,1186.194483,12929.650000,3600.0,180.0,1,2,54,2.0,0,1
384741,3260593818,1,1,2024,28524.56,0.0,0.0,0.0,0.0,21359.691533,...,332.148276,8406.885000,0.0,337.5,2,2,63,2.0,0,0
